In [1]:
from datetime import datetime
import pandas as pd
import xml.etree.ElementTree as ET

from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, 
    FloatType, DateType)

# Entity attribute columns
ENTITY_COLS = ['LegalName',  'OtherEntityNames', 'TransliteratedOtherEntityNames',
    'LegalAddress_FirstAddressLine', 'LegalAddress_AdditionalAddressLine', 'LegalAddress_City', 
    'LegalAddress_Country', 'LegalAddress_PostalCode',
    'HeadquartersAddress_FirstAddressLine', 'HeadquartersAddress_AdditionalAddressLine', 'HeadquartersAddress_City', 
    'HeadquartersAddress_Country', 'HeadquartersAddress_PostalCode',
    'OtherAddresses_FirstAddressLine', 'OtherAddresses_AdditionalAddressLine', 'OtherAddresses_City', 
    'OtherAddresses_Country', 'OtherAddresses_PostalCode',
    'TransliteratedOtherAddresses_FirstAddressLine', 'TransliteratedOtherAddresses_AdditionalAddressLine', 
    'TransliteratedOtherAddresses_City', 
    'TransliteratedOtherAddresses_Country', 'TransliteratedOtherAddresses_PostalCode',
    'RegistrationAuthorityID', 'RegistrationAuthorityEntityID',
    'LegalJurisdiction',
    'EntityCategory',
    'EntityLegalFormCode', 'OtherLegalForm',
    'AssociatedLEI',
    'EntityStatus',
    'EntityExpirationDate',
    'EntityExpirationReason',
    'SuccessorLEI'
]

# Entity registration attribute columns
REGISTRATION_COLS = ['InitialRegistrationDate', 'LastUpdateDate', 'RegistrationStatus',
    'NextRenewalDate', 'ManagingLOU', 'ValidationSources',
    'ValidationAuthority', 
    'OtherValidationAuthorities'
]

# Other attribute columns
OTHER_COLS = ['LEI',]

# Auxiliary column lists
ADDRESS_COLS = [ 'FirstAddressLine', 'AdditionalAddressLine', 'City', 'Country', 'PostalCode']
REG_AUTHORITY_COLS = ['RegistrationAuthorityID', 'RegistrationAuthorityEntityID']
LEGAL_FORM_COLS = ['EntityLegalFormCode', 'OtherLegalForm']
ASSOCIATED_ENTITY_COLS = ['AssociatedLEI']
SUCCESSOR_ENTITY_COLS = ['SuccessorLEI']

BASIC_REGISTRATION_COLS = ['InitialRegistrationDate', 'LastUpdateDate', 'RegistrationStatus',
    'NextRenewalDate', 'ManagingLOU', 'ValidationSources']

VAL_AUTHORITY_COLS = ['ValidationAuthorityID', ]

# TODO: decide on OtherValidationAuthorities and EXTENSIONS

# columns for the DataFrame
COL_NAMES = OTHER_COLS + ENTITY_COLS + REGISTRATION_COLS

def set_schema():
    """
    Define the schema for the DataFrame
    """
    schema_list = []
    for c in COL_NAMES:
        schema_list.append(StructField(c, StringType(), True))
        #if c == 'price':
        #    schema_list.append(StructField(c, FloatType(), True))
        #if c == 'InitialRegistrationDate':
        #    schema_list.append(StructField(c, DateType(), True))
        #elif c == 'LastUpdateDate':
        #    schema_list.append(StructField(c, DateType(), True))
        #elif c == 'NextRenewalDate':
        #    schema_list.append(StructField(c, DateType(), True))
        #elif c == 'EntityExpirationDate':
        #    schema_list.append(StructField(c, DateType(), True))
        #else:
        #    schema_list.append(StructField(c, StringType(), True))
    
    return StructType(schema_list)

def parse_xml(rdd):
    """
    Read the xml string from rdd, parse and extract the elements,
    then return a list of list.
    """
    results = []
    root = ET.fromstring(rdd[0])
    
    lei_records = root.find('LEIRecords')

    for b in lei_records.findall('LEIRecord'):
        rec = []
        # Extract LEI ID
        e = 'LEI'
        if b.find(e) is None:
            rec.append(None)
        else:
            value = b.find(e).text
            rec.append(value)
        
        # parse entity attributes
        e = 'Entity'
        entity = b.find(e)
        if entity is None:
            for c in ENTITY_COLS:
                rec.append(None)
        else:
            # parse legal address details
            address = entity.find('LegalAddress')
            if address is None:
                for el in ADDRESS_COLS:
                    rec.append(None)
            else:
                for el in ADDRESS_COLS:
                    if address.find(el) is None:
                        rec.append(None)
                    else:
                        value = address.find(el).text
                        rec.append(value)
            # parse Headquarter address details
            address = entity.find('HeadquartersAddress')
            if address is None:
                for el in ADDRESS_COLS:
                    rec.append(None)
            else:
                for el in ADDRESS_COLS:
                    if address.find(el) is None:
                        rec.append(None)
                    else:
                        value = address.find(el).text
                        rec.append(value)
            # parse Other address details
            address = entity.find('OtherAddresses')
            if address is None:
                for el in ADDRESS_COLS:
                    rec.append(None)
            else:
                for el in ADDRESS_COLS:
                    if address.find(el) is None:
                        rec.append(None)
                    else:
                        value = address.find(el).text
                        rec.append(value)
            # parse TransliteratedOtherAddresses address details
            address = entity.find('TransliteratedOtherAddresses')
            if address is None:
                for el in ADDRESS_COLS:
                    rec.append(None)
            else:
                for el in ADDRESS_COLS:
                    if address.find(el) is None:
                        rec.append(None)
                    else:
                        value = address.find(el).text
                        rec.append(value)
        
            # parsing RegistrationAuthority
            e = 'RegistrationAuthority'
            reg_auth = entity.find(e)
            if reg_auth is None:
                for el in REG_AUTHORITY_COLS:
                    rec.append(None)
            else:
                for el in REG_AUTHORITY_COLS:
                    if reg_auth.find(el) is None:
                        rec.append(None)
                        continue
                    value = reg_auth.find(el).text
                    rec.append(value)
        
            # parsing LegalJurisdiction
            e = 'LegalJurisdiction'
            legal_jur = entity.find(e)
            if legal_jur is None:
                rec.append(None)
            else:
                value = legal_jur.text
                rec.append(value)
        
            # parsing EntityCategory
            e = 'EntityCategory'
            entity_cat = entity.find(e)
            if entity_cat is None:
                rec.append(None)
            else:
                value = entity_cat.text
                rec.append(value)
        
            # parsing LegalForm
            e = 'LegalForm'
            legal_form = entity.find(e)
            if legal_form is None:
                for el in LEGAL_FORM_COLS:
                    rec.append(None)
            else:
                for el in LEGAL_FORM_COLS:
                    if legal_form.find(el) is None:
                        rec.append(None)
                        continue
                    value = legal_form.find(el).text
                    rec.append(value)
        
            # parsing AssociatedEntity
            e = 'AssociatedEntity'
            associated_entity = entity.find(e)
            if associated_entity is None:
                for el in ASSOCIATED_ENTITY_COLS:
                    rec.append(None)
            else:
                for el in ASSOCIATED_ENTITY_COLS:
                    if associated_entity.find(el) is None:
                        rec.append(None)
                        continue
                    value = associated_entity.find(el).text
                    rec.append(value)
        
            # parsing EntityStatus
            e = 'EntityStatus'
            entity_status = entity.find(e)
            if entity_status is None:
                rec.append(None)
            else:
                value = entity_status.text
                rec.append(value)
        
            # parsing EntityExpirationDate
            e = 'EntityExpirationDate'
            entity_exp_date = entity.find(e)
            if entity_exp_date is None:
                rec.append(None)
            else:
                value = entity_exp_date.text
                value = datetime.strptime(value, '%Y-%m-%d') # cast to datetime
                rec.append(value)
        
            # parsing EntityExpirationReason
            e = 'EntityExpirationReason'
            entity_exp_reason = entity.find(e)
            if entity_exp_reason is None:
                rec.append(None)
            else:
                value = entity_exp_reason.text
                rec.append(value)
        
            # parsing SuccessorEntity
            e = 'SuccessorEntity'
            successor_entity = entity.find(e)
            if successor_entity is None:
                for el in SUCCESSOR_ENTITY_COLS:
                    rec.append(None)
            else:
                for el in SUCCESSOR_ENTITY_COLS:
                    if successor_entity.find(el) is None:
                        rec.append(None)
                        continue
                    value = successor_entity.find(el).text
                    rec.append(value)
        
        # parse Registration attributes
        e = 'Registration'
        registration = b.find(e)
        if registration is None:
            for c in REGISTRATION_COLS:
                rec.append(None)
        else:
            # parsing basic registration attributes
            for el in BASIC_REGISTRATION_COLS:
                if registration.find(el) is None:
                    rec.append(None)
                    continue
                value = registration.find(el).text
                if el in ['InitialRegistrationDate', 'LastUpdateDate', 'NextRenewalDate']:
                    value = datetime.strptime(value, '%Y-%m-%d') # cast to datetime
                rec.append(value)
            
            # parsing ValidationAuthority
            e = 'ValidationAuthority'
            registration_auth = registration.find(e)
            if registration_auth is None:
                for c in VAL_AUTHORITY_COLS:
                    rec.append(None)
            else:
                for el in VAL_AUTHORITY_COLS:
                    if registration_auth.find(el) is None:
                        rec.append(None)
                        continue
                    value = registration_auth.find(el).text
                    rec.append(value)
        
        results.append(rec)

    return results

In [2]:
# define the schema
my_schema = set_schema()
# read each xml file as one row, then convert to RDD
file_rdd = spark.read.text("gs://lei-data/20210504-l1-data/20210504-gleif-concatenated-file-lei2.xml", wholetext=True).rdd
# parse xml tree, extract the records and transform to new RDD
records_rdd = file_rdd.flatMap(parse_xml)
print("Parsed XML")
# convert RDDs to DataFrame with the pre-defined schema
df = records_rdd.toDF(my_schema)
print("RDD to DataFrame converted")
df.printSchema()

Parsed XML
RDD to DataFrame converted
root
 |-- LEI: string (nullable = true)
 |-- LegalName: string (nullable = true)
 |-- OtherEntityNames: string (nullable = true)
 |-- TransliteratedOtherEntityNames: string (nullable = true)
 |-- LegalAddress_FirstAddressLine: string (nullable = true)
 |-- LegalAddress_AdditionalAddressLine: string (nullable = true)
 |-- LegalAddress_City: string (nullable = true)
 |-- LegalAddress_Country: string (nullable = true)
 |-- LegalAddress_PostalCode: string (nullable = true)
 |-- HeadquartersAddress_FirstAddressLine: string (nullable = true)
 |-- HeadquartersAddress_AdditionalAddressLine: string (nullable = true)
 |-- HeadquartersAddress_City: string (nullable = true)
 |-- HeadquartersAddress_Country: string (nullable = true)
 |-- HeadquartersAddress_PostalCode: string (nullable = true)
 |-- OtherAddresses_FirstAddressLine: string (nullable = true)
 |-- OtherAddresses_AdditionalAddressLine: string (nullable = true)
 |-- OtherAddresses_City: string (nulla

In [3]:
df.limit(1).show()

Py4JJavaError: An error occurred while calling o78.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 4 times, most recent failure: Lost task 0.3 in stage 0.0 (TID 3) (dev-cluster-w-1.europe-west1-b.c.lorelee-keep-312708.internal executor 4): ExecutorLostFailure (executor 4 exited caused by one of the running tasks) Reason: Container from a bad node: container_1620767273846_0015_01_000004 on host: dev-cluster-w-1.europe-west1-b.c.lorelee-keep-312708.internal. Exit status: 52. Diagnostics: [2021-05-12 16:45:09.392]Exception from container-launch.
Container id: container_1620767273846_0015_01_000004
Exit code: 52

[2021-05-12 16:45:09.394]Container exited with a non-zero exit code 52. Error file: prelaunch.err.
Last 4096 bytes of prelaunch.err :
Last 4096 bytes of stderr :
t writer for /opt/conda/miniconda3/bin/python
java.lang.OutOfMemoryError: input is too large to fit in a byte array
	at org.sparkproject.guava.io.ByteStreams.toB

In [ ]:
# Saving the data to BigQuery
# 'com.google.cloud.spark.bigquery'
table = "lorelee-keep-312708.lei_staging.entities"
df.write.format('bigquery') \
  .option("temporaryGcsBucket","temp-dataproc-storage") \
  .option('table', table) \
  .mode("overwrite") \
  .save()

print("Finished the workflow")